# SVMs_processing_models

Data: 04/05

# Imports and dependencies

In [ ]:
import numpy as np
import pandas as pd
import sys
import os
from pymfe.mfe import MFE
import csv
from sources.models import *
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.impute import KNNImputer
#import scipy.spatial.distance as dist
from sklearn.preprocessing import StandardScaler, LabelEncoder
from typing import Any
import traceback
import sources.cat_attrib as cat_attrib
import seaborn as sns
import matplotlib.pyplot as plt
import time

c:\Users\user\anaconda3\envs\AC1_WIP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Files and Paths

In [ ]:
import config
from config import *
sys.path.insert(0, os.getcwd())

import sources.compare as compare# construção de ficheiros de output

In [ ]:
log_path: Path = compare.get_output_file(function_name='pipeline_log', output_name="pipe_log")

#Configuração do ficheiro que guarda o sucesso/insucesso de cada operação
logging.basicConfig(
    filename=str(log_path),
    filemode='a',
    level=logging.INFO,
    force=True, #Com cada novo run-cycle, criamos um novo ficheiro pipe_log
    format='%(asctime)s — %(levelname)s — %(message)s',
    datefmt='%H:%M:%S %Y-%m-%d'
)

# Interlink jupyter processing + models

Para acedermos às classes definidas em models.py e executar os notebooks individualmente;

Para

# Kernel template classes

O método __call__ dá o template da função a ser aplicada pelos diferentes objetos Linear(), Poly() ou RBF() criados na implementação. Estes objetos kernel são gerados no fit() a partir destas classes modelo;

In [ ]:

"""Kernel"""
class Linear(object):
    def __call__(self, x, y):
        return np.dot(x, y.T)

    def __repr__(self):
        return "Linear kernel"


class Poly(object):
    def __init__(self, degree=2):
        self.degree = degree

    def __call__(self, x, y):
        return np.dot(x, y.T) ** self.degree

    def __repr__(self):
        return "Poly kernel"

class RBF(object):
    def __init__(self, gamma=1.0):
        self.gamma = gamma
        # Higher gamma, extreme model fitting (overfitting)
    def __call__(self, x,y):
        #Computação da norma do vetor ao quadrado
        #Computação usando broadcasting do numpy torna isto mais eficiente.
        diff = x-y # está a correr infinitamente --> tenhos de ver :()
        return np.exp(-self.gamma * np.sum(diff**2, axis=-1))
    def __repr__(self):
        return f"RBF kernel (gamma={self.gamma})"

np.random.seed(9999)


Método computation_limit é herdado da classe BaseEstimator.

Aplicada em todos os métodos;
De acordo com a comparação "diff < tol" e max_iter, interrompemos o SVM quando o modelo continua a calcular erros a partir de pontos aleatórios na matriz, mas sem melhorar as margens do hiperplano.

Caso seja ativado, estimador deve retornar 0,0,0 para append no dataset_summary.

---

Adicionei a condiçao KKT ao modified_3 e modified_3_1 para que a escolha de pares de pontos responda às limitações.

# Classe Dataset

In [ ]:

class Dataset:

        def __init__(self, data: pd.DataFrame, name: str | None=None): #Definimos name como atributo opcional
            self.name = name
            self.data = data
            self.X = pd.DataFrame()
            self.y = pd.Series()
            self.attrib_dict = {}
            self.metafeatures_dict = {}
            self.nominal_cols = []
            self.pct_categorical = 0
            self.quantitative_cols=[]

        def split(self):
            self.X = self.data.drop(columns=[self.data.columns[-1]])#features
            self.y = self.data[self.data.columns[-1]]

In [ ]:
class Processing:
    noise_dict = {}
    one_hot_encoding_dict = []

    KERNEL_MAP = {
        'linear': Linear(),
        'poly': Poly(degree=2),
        'rbf': RBF(gamma=0.1)
    }

    def __init__(self, processing_input_path=config.datasets_path, processed_dataset_output_path=config.processed_dataset_output_path, model=None): #model é um objeto importado do config: Standard, (...)
        self.path = processing_input_path
        self.model = model  # 1. ATRIBUIR PRIMEIRO
        # MFE_flag removed: metafeature extraction is now mandatory in the pipeline
        self.processed_dataset_output = processed_dataset_output_path
        self.processed_datasets: dict[str, dict[str, Any]] = {}
        self.metafeatures_dict_list: list[dict] = []
        self.header_flag = True
        self.metafeatures_file_path: Path = compare.get_output_file(function_name="metafeatures", output_name="meta")
        self.__main__()     # 2. EXECUTAR DEPOIS


    def __main__(self):
        if not os.path.exists(self.path):
            print(f"Erro: O diretório {self.path} não foi encontrado.")
            return

        files = [f for f in os.listdir(self.path) if f.endswith('.csv')]

        # Truncate previous output if it exists (metafeatures always extracted)
        if self.processed_dataset_output and os.path.isfile(self.processed_dataset_output):
            with open(self.processed_dataset_output, "w") as f:
                f.truncate()


        # Ao invés de criar uma instância limpa, referencia o model. A adição  do kernel_obj, noise_level é feita antes da construção do estimator no train and avaluate
        model_to_use = self.model


        for file_name in files:
            full_path = os.path.join(self.path, file_name)
            try:
                df = pd.read_csv(full_path)
                current_dataset = Dataset(data=df, name=file_name)
                current_dataset.split()

                # --- Metafeature extraction on RAW data (before OHE/scaling/feature selection) ---
                # Runs unconditionally: metafeatures are required for kernel selection and
                # outlier-aware SVM model behaviour. Must precede processing to preserve
                # original data distribution for Mahalanobis-based outlier computation.
                self.metafeatures_extraction(dataset=current_dataset)


                counts = pd.Series(current_dataset.y).value_counts()
                imbalance_ratio = counts.max() / counts.min()
                current_dataset.metafeatures_dict['imbalance_ratio'] = imbalance_ratio  # threshold you can tune


                chosen_kernel_str = self.best_kernel(current_dataset)
                print(f"[{current_dataset.name}] DNA sugere: {chosen_kernel_str}")

                # --- Processing pipeline (runs on post-raw data) ---
                self.one_hot_encoding(current_dataset)
                self.scaling(current_dataset)
                self.NaN_KNN_filling(current_dataset)
                self.check_noise_column(current_dataset)
                self.calc_correlation(current_dataset)

                kernel_obj = self.KERNEL_MAP.get(chosen_kernel_str, 'linear')

                #calcular noise factor:
                noise_level = current_dataset.attrib_dict.get('ns_ratio', 0.0)


                # Guardar resultados
                self.processed_datasets[current_dataset.name] = {
                    'X': np.array(current_dataset.X, dtype=np.float64),
                    'y': np.array(current_dataset.y, dtype=np.int32),
                    'attrib_dict': current_dataset.attrib_dict,
                    'metafeatures_dict': current_dataset.metafeatures_dict,
                    'name': current_dataset.name,
                    'kernel_used': chosen_kernel_str,
                    'kernel_obj' : kernel_obj
                }

            except Exception as e:
                print(f"Não foi possível processar {file_name}: {e}")
                traceback.print_exc()

        self.train_and_evaluate(processed_datasets=self.processed_datasets, model_class=model_to_use)


    def NaN_KNN_filling(self, dataset: Dataset):

        #Elimina colunas totalmente preenchidas por valores NaN
        dataset.X = dataset.X.dropna(axis=1, how='all')

        #Separar colunas em categóricas e numéricas. Estes 2 tipos de dados têm encoding distinto
        """numerical_columns: list = dataset.quantitative_cols
        categorical_columns: list = [col for col in dataset.nominal_cols if col in dataset.X.columns]
        """

        categorical_columns = dataset.X.select_dtypes(exclude=[np.number]).columns.tolist()
        numerical_columns = dataset.X.select_dtypes(include=[np.number]).columns.tolist()

        # missing values em colunas não categoricas
        imputer = KNNImputer(n_neighbors=5)
        if len(numerical_columns) > 0:
            dataset.X[numerical_columns] = imputer.fit_transform(dataset.X[numerical_columns])


        # missing values em colunas categoricas
        if len(categorical_columns) > 0:
            for col in categorical_columns:
                dataset.X[col] = dataset.X[col].fillna(dataset.X[col].mode()[0])

        #--Check---#
        try:
            assert dataset.X.isna().sum().sum() == 0
            logging.info(f"[{dataset.name}] NaN filling - SUCCESS")

        except AssertionError:
            logging.error(f"[{dataset.name}] NaN filling - FAILED")

        """
        snapshot = pd.concat(
            [dataset.X, pd.Series(dataset.y, name='target')],
            axis=1
        )

        prefixed_header = [f"{dataset.name}__{col}" for col in snapshot.columns]

        snapshot.to_csv("NanFilling.csv", index=False, mode="a", header=prefixed_header)
        """


        return dataset.X


    def one_hot_encoding(self, dataset: Dataset):
       #Este é a lista de colunas categoricas pos remoção de colunas preenchidas por NaN dataset. No processamento, só em NaN_filling é que estas colunas são removidas no dataset original

        cols_to_encode = [col for col in dataset.nominal_cols if col in dataset.X.columns]

        dataset.X = pd.get_dummies(dataset.X, columns=cols_to_encode, dtype=int)

        le = LabelEncoder()
        # Convertemos o resultado do fit_transform (array) de volta para Series
        encoded_y = le.fit_transform(dataset.y)
        dataset.y = pd.Series(encoded_y, index=dataset.X.index, name='target')
        dataset.y = dataset.y.replace(0, -1)

        #--Check---#
        #A classe target (y) só pode adquirir os valores -1,1
        try:
            assert set(dataset.y.unique()).issubset({-1, 1}), f"Target classification used unexpected labels {dataset.y.unique()}"
            logging.info("O.H.E SUCCESS")
        except AssertionError as e:
            logging.error(f"[{dataset.name}] Encoding failed: {e}")


        snapshot = pd.concat(
            [dataset.X, pd.Series(dataset.y, name='target')],
            axis=1
        )

        prefixed_header = [f"{dataset.name}__{col}" for col in snapshot.columns]

        snapshot.to_csv("ohe_temp.csv", index=False, mode="a", header=prefixed_header)

    #Normalização
    def scaling(self, dataset: Dataset):

        quant_cols_survivor = [
            col for col in dataset.quantitative_cols
            if col in dataset.X.columns
        ]

        #Saltamos o dataset se for totalmente composto por atributos categoricos (dataset_24)
        if not quant_cols_survivor:
            logging.warning(f"[{dataset.name}] scaling — no quantitative columns "
                            f"to scale. Dataset is likely fully categorical, skipping.")
            return

        #Fitting só acontece no conjunto de atributios preditivos. Evitamos Leakage
        scaler = StandardScaler()
        dataset.X[quant_cols_survivor] = scaler.fit_transform(
            dataset.X[quant_cols_survivor]
        )

        #--Check---#
        #Os resultados têm de seguir uma distribuição Normal N(0,1), logo, \miu = X.mean() = 0 e \sd = X.std() = 1
        try:
            assert dataset.X.mean().abs().max() < 1e-6
            assert (dataset.X.std() - 1).abs().max() < 1e-6
            logging.info(f"[{dataset.name}] scaling - SUCCESS")
        except AssertionError:
            logging.error(f"[{dataset.name}] scaling — FAILED")


        snapshot = pd.concat(
            [dataset.X, pd.Series(dataset.y, name='target')],
            axis=1
        )

        prefixed_header = [f"{dataset.name}__{col}" for col in snapshot.columns]

        snapshot.to_csv("scaletemp.csv", index=False, mode="a", header=prefixed_header)



    # Drop Zero-Variance Features
    def check_noise_column(self, dataset: Dataset, threshold: int=10):
        cols = []
        for column in dataset.data.columns:

            #dropna: o parametro dropna=False garante que colunas com NaN são contadas como colunas zero-variance
            freq_list = dataset.data[column].value_counts(normalize=True, dropna=False).sort_values(ascending=False)

            #Elimina colunas com 1 único valor único (vero-variance features)
            if len(freq_list) < 2:
                cols.append(column)
                continue

            ratio = freq_list.iloc[0] / freq_list.iloc[1]
            if (ratio > threshold):
                cols.append(column)

        dataset.data.drop(columns=cols, inplace=True)



    def calc_correlation(self, dataset: Dataset):

        df_temp = dataset.X.copy()
        target_name = "target_class_temp"
        df_temp[target_name] = dataset.y

        threshold_target = 0.01  # Correlação mínima com alvo
        threshold_multicolinearity = 0.9  # Correlação máxima entre features

        corr_matrix = df_temp.corr().abs() #pd.Dataframe()
        target_corr = corr_matrix[target_name].drop(labels=[target_name])

        relevant_features = target_corr[target_corr > threshold_target].index.tolist() #lista de nomes de atributos relevantes (acima do limite mínimo)

        if not relevant_features:
            # Caso nenhuma feature seja relevante, mantém-se as originais para não esvaziar o dataset
            relevant_features = dataset.X.columns.tolist()

        # 3. Remover multicolinearidade entre as features selecionadas
        df_features = df_temp[relevant_features]
        corr_features = df_features.corr().abs() # calculamos pd.Dataframe de correlação para as features relevantes.

        #Todos os valore que satisfazem a condição booleana "True" (filtro do array booleano), os seus valores de correlação são
        #guardados na área acima da linha diagonal principal da matriz, ou seja, obtemos uma matriz com o triangulo superior preenchido de valores de correlação positivos
        #e o triangulo inferior preenchido por NaNs.
        upper = corr_features.where(np.triu(np.ones(shape= corr_features.shape), k=1).astype(bool))


        #comparamos todas as células da matriz de valores de correlação que satisfizeram o limite threshold com o limite de multicolinariedade.
        #gera uma lista em compreensão com os índices das listas que satisfazem esta condição
        to_drop = [col for col in upper.columns if any(upper[col] > threshold_multicolinearity)]

        #mantemos apenas as colunas que passaram os 2 "filtros", ou seja, estão contidos no intervalo [0.01, 0.9] .
        final_features = [f for f in relevant_features if f not in to_drop]

        # Guardamos apenas os atributos preditivos do dataframe original que pertencem à lista de strings criada.
        dataset.X = dataset.X[final_features]

        dataset.attrib_dict['selected_features'] = final_features

        #---Check-##




    def train_and_evaluate(self, processed_datasets: dict, model_class=None):

        if model_class==SVM_STANDARD:
            output_name = "eval_SVM_STANDARD.csv"

        if model_class==SVM_modified_3:
            output_name = "eval_SVM_modified_3.csv"
        
        if model_class==SVM_modified_3_ponto_1:
            output_name = f"eval_SVM_modified_3_1.csv"

        #output_path = compare.get_output_file(function_name=self.train_and_evaluate.__name__, output_name=output_name)
        output_path = config.final_eval_file / f"{output_name}"
        print(output_path)

        # Use the provided class, or fall back to self.model (always a class, never an instance)
        model_cls = model_class if model_class is not None else self.model
        if model_cls is None:
            model_cls = SVM_STANDARD  # fallback to the class itself, not an instance

        scoring = {
            'acc': 'balanced_accuracy',
            'f1': 'f1_weighted',
            'precision': 'precision_weighted'
        }

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        dataset_summary = []
        for dataset in processed_datasets.values():

            print(f"\n--- Avaliando: {dataset['name']} ---")
            start_time = time.time()


            kernel_obj = dataset.get('kernel_obj', Linear())
            noise_level = dataset.get('attrib_dict', {}).get('ns_ratio', 0.0)
            estimator = model_cls(kernel=kernel_obj, C=1.0, noise_factor=noise_level)

            estimator.dataset_name = dataset['name']

            try:
                # Executa a validação cruzada
                # error_score='raise' ajuda-nos a ver o erro real se algo falhar no fit

                cv_results: dict = cross_validate(
                    estimator= estimator,
                    X = dataset["X"],
                    y = dataset["y"],
                    cv=skf, #BaseCrossValidator
                    scoring=scoring, #Métricas selecionadas para avaliar performance do modelo validado no dataset input.
                    return_train_score=False,
                    error_score='raise'
                )

                """

                CV_RESULTS: dicionario agrupa os valores calculados por dobra numa lista associada ao
                atributo de classificação.

                """

                # 3. Aceder às chaves corretas: o sklearn adiciona 'test_' à frente da chave do dicionário
                acc_mean = cv_results['test_acc'].mean()
                f1_mean = cv_results['test_f1'].mean()
                precision_mean= cv_results['test_precision'].mean()

                print(f"Resultados:")
                print(f"* Accuracy: {acc_mean:.4f} ± {cv_results['test_acc'].std():.4f}")
                print(f"* F1-Score: {f1_mean:.4f} ± {cv_results['test_f1'].std():.4f}")
                print(f"* Precision: {precision_mean:.4f} ± {cv_results['test_precision'].std():.4f}")


                dataset_summary.append(
                    {
                        "Dataset": dataset['name'],
                        "Model": estimator.__class__.__name__,
                        "Kernel Function": dataset["kernel_used"],
                        "Balanced Accuracy": f"{acc_mean:.4f} +/- {cv_results['test_acc'].std():.4f}",
                        "F1-Score (Macro)": f"{f1_mean:.4f} +/- {cv_results['test_f1'].std():.4f}",
                        "Precision": f"{precision_mean:.4f} +/- {cv_results['test_precision'].std():.4f}"
                    }
                )

            except Exception as e:
                print(f"❌ Erro crítico no treino: {e}")
                traceback.print_exc()

            end_time = time.time()
            duration = end_time - start_time
            print(f"TRAIN_EVALUATE_TIME \t Dataset: {dataset['name']} Model: {estimator.__class__.__name__} TRAIN_EVALUATE_TIME: {duration} \n")
            print()

        compare.building_table(file_path= output_path, summary=pd.DataFrame(dataset_summary))#Guardar os resultados num .csv

    def best_kernel(self, dataset: Dataset):

        """
        Analisa DNA do dataset (metafeatures_dict) para perceber qual é o kernel mais indicado para
        problemas de outliers (kurtosis e skewness), linearity (linear_discr)

        """

        dna = dataset.metafeatures_dict
        highly_categorical = dataset.pct_categorical > 0.8



        #Extração de metafeatures

        d = dna.get("nr_attr") #Dimensionalidade do dataset
        n = dna.get("nr_inst")
        skewness = dna.get("skewness.nanmean") or 0.0 #Salva-guarda para evitar erros de computação ou valores sem sentido.
        kurtosis = dna.get("kurtosis.nanmean") or 0.0
        linear_discr = dna.get("linear_discr.nanmean") or 0.0
        noise = dna.get("ns_ratio") or 0.0

        imbalance_ratio = dna.get("imbalance_ratio",1.0)

        #Projeção da proporção de desequilíbiro de classe na escala logarítmica.
        #Facilita definir um threshold geral (>0.5) para considerar o dataset desequilibrado

        imbalance_score = min(np.log(imbalance_ratio) / np.log(20), 1.0)


        """Definimos 3 problemas possíveis
            Elevada dimensionalidade
            Datasets com má separação linear
            Datasets ricos em atributos sem variância
            Datasets ricos em outliers
        """
        # Definimos um máximo que o score pode adquir: 1.0  #

        scores = {

            "high_dim":         min(d / max(n, 1), 1.0),
            "low_separability": 0.0 if highly_categorical else 1.0 - min(linear_discr, 1.0),
            "noisy": min((noise or 0.0) / 0.2, 1.0),
            "outlier_heavy":    0.0 if highly_categorical else min((abs(kurtosis) + skewness) / 10.0, 1.0),
            "class_imbalance":  0.0 if highly_categorical else (imbalance_score if imbalance_score > 0.5 else 0.0),#ratio 5:2, em log(20), é 0.54
        }

        MODEL_TABLE = [
            ("linear", {"high_dim":0.9,"low_separability":0.2}),
            ("poly", {"outlier_heavy":0.8,"low_separability":0.6}),
            ("rbf", {"noisy": 0.9, "outlier_heavy":0.5, "class_imbalance":0.7})
        ]

        """
        ranked_problems

        key parameter da função sorted()

        Somatório em linha dos scores dos problemas para cada dataset.

        Retorna o kernel adequado para cada problema do dataset, organizando a lista de tuplos
        MODELO, SCORE DATASET por ordem decrescente de importância.

        """


        ranked_problems: list[str|dict] = sorted(
            [
                (kernel, weights, sum(scores.get(r, 0) * w for r, w in weights.items()))
                for kernel, weights in MODEL_TABLE
            ],
            key=lambda row: row[2],  # sort by the computed score (3rd element)
            reverse=True
        )

        #Escolhemos o
        kernel_chosen = ranked_problems[0][0]


        print(f"dataset_name: {dataset.name}\n")
        print(f"ranked_problems: {ranked_problems}\n")
        print(f"categorical percentage (0.8): {dataset.pct_categorical}\n")
        print(f"kernel_chosen: {kernel_chosen}\n")
        print("\n" + "-"*20 + "\n")

        return kernel_chosen

    def metafeatures_extraction(self, dataset: Dataset)-> str:

        #-Gerar ficheiro output 1 única vez por run cycle
        #Criamos copias temporarias dos datasets para não afetar os dados originais.
            # Este processamento é contido ao método. Para que MFE possa extrair metadados, só podem existir
            # valores numéricos e SEM missing values

        try:
            X_temp = dataset.X.copy()

            #Eliminamos as colunas zero-variance antes de retirarmos os indices das colunas nominas (cat_indices)
            #Evitamos desfazamento entre o índice verdadeiro da coluna e o indice guardado, pq depois da eliminação o númerko de atributos será diferente.
            X_temp = X_temp.dropna(axis=1, how='all')

            dataset.quantitative_cols, dataset.nominal_cols, dataset.pct_categorical = cat_attrib.get_categorical_indicator(dataset_name=dataset.name)

            #Fill NaNs on categorical columns
            for col in X_temp.columns:
                if col in dataset.nominal_cols: #Se a coluna for nominal categórica (diag: 1,2,3) preenchemos missing values com moda.
                    X_temp[col] = X_temp[col].fillna(X_temp[col].mode()[0])
                else:
                    X_temp[col] = X_temp[col].fillna(X_temp[col].median())


            cat_indices = [i for i,col in enumerate(X_temp.columns)if col in dataset.nominal_cols]


            X_no_NaN = X_temp.to_numpy()


            le_temp = LabelEncoder()
            y_encoded = le_temp.fit_transform(dataset.y.fillna(dataset.y.mode()[0])) #fillna retorna um novo array sem alterar dataset.y original

            print(f"[{dataset.name}] nominal_cols from OpenML: {dataset.nominal_cols}")
            print(f"[{dataset.name}] numeric cols in X_temp: {X_temp.select_dtypes(include='number').columns.tolist()}")
            print(f"[{dataset.name}] remaining NaN after fill: {X_temp.isna().sum().sum()}")


            mfe = MFE(
                    groups=["general", "statistical", "info-theory", "complexity", "landmarking"],
                    features=[
                        "nr_attr",
                        "nr_inst",
                        "sparsity",
                        "skewness",
                        "mut_inf",
                        "linear_discr",
                        #"nr_outliers",    # ← direct outlier count Não utilizamos nr_outliers., porque se MFE não consegue computar outliers, outras métricas dependentes de LOF não são computados (input de missing values)-
                        "kurtosis",    # ← distribution tail weight
                        "iq_range"    # ←spread for IQR rule
                    ],
                    summary=["nanmean", "nansd"],
                    #Local Outliewr Factor, utlilizado pelo PymFE, define que n_vizinhos < n_amostras. n_neighbors=20, logo, um dataset com apenas 21
                    # degenerad distâncias e retorna NaN values
                )

            """mean, sd não ignora colunas zero-variance, logo, kurtosis e skewness não poderão ser calculados corretamente.
            nanmean is the mean summary function which ignores all nan values" (documentação pymfe)"""

            mfe.fit(X_no_NaN, y_encoded, cat_cols=cat_indices)
            names, values = mfe.extract()


            dataset.metafeatures_dict = {
                    k: v for k, v in zip(names, values)
                    if not (isinstance(v, float) and np.isnan(v))
                }

            self.metafeatures_dict_list.append(dataset.metafeatures_dict)


            with open(self.metafeatures_file_path, 'a', newline='') as f:

                writer = csv.writer(f)
                if self.header_flag:
                    header = ["dataset_name"] + names
                    writer.writerow(header)
                    self.header_flag=False

                row = [dataset.name] + list(values)
                writer.writerow(row)

            print(f"{dataset.name}: Metafeature extraction completed")

        except Exception as e:
            print(f"{dataset.name}: Unexpected failure — {type(e).__name__}: {e}")
            traceback.print_exc()


In [ ]:
data_test_path = base_dir / "noise_outliers"

t1=Processing(processing_input_path=data_test_path, model=SVM_STANDARD)
t2=Processing(processing_input_path=data_test_path, model=SVM_modified_3)
t3=Processing(processing_input_path=data_test_path, model=SVM_modified_3_ponto_1)

logging.shutdown() #Para fechar o ficheiro e torná-lo eliminável

--- Avaliando: dataset_802_pbcseq.csv ---
Resultados:
* Accuracy: 0.6460 ± 0.0349
* F1-Score: 0.6235 ± 0.0545
* Precision: 0.6901 ± 0.0316
TRAIN_EVALUATE_TIME 	 Dataset: dataset_802_pbcseq.csv Model: SVM_modified_3_ponto_1 TRAIN_EVALUATE_TIME: 71.76656436920166

--- Avaliando: dataset_802_pbcseq.csv ---
Resultados:
* Accuracy: 0.5269 ± 0.0438
* F1-Score: 0.3992 ± 0.1054
* Precision: 0.6611 ± 0.0938
TRAIN_EVALUATE_TIME 	 Dataset: dataset_802_pbcseq.csv Model: SVM_modified_3 TRAIN_EVALUATE_TIME: 7.554677486419678

